# LushProtein — Solution 2: Four-Layer Recommendation Engine (Reproducible Analysis)



**What this notebook reproduces:**
1. Problem-statement metrics (one-and-done rate, category ladder, subscriber gap)
2. **Layer 1** — rule-based cold start (category co-purchase matrix)
3. **Layer 2** — market basket analysis (same-order SKU association rules)
4. **Layer 3** — timed post-purchase cross-sell & sample schedule
5. **Layer 4** — item-item collaborative filtering
6. CRM treatment tiers and incentive budgets
7. Presentation charts under `EDA/aditya_findings/outputs/charts/`

**Data:** Pre-cleaned finals in `EDA/outputs_finals/` (built by `EDA/13_build_finals_datasets.py`). No ad-hoc cleaning in this notebook — we load finals and validate against `manifest.json`.

 Requires: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`, `openpyxl`, `pyarrow`.


## 0. Setup & path resolution

In [1]:
import warnings
warnings.filterwarnings("ignore")

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


def ensure_deps(*packages: str) -> None:
    """Install missing notebook dependencies into the active kernel."""
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *missing],
            stdout=subprocess.DEVNULL,
        )


ensure_deps("openpyxl", "pyarrow")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

# ── Resolve project root (notebook lives at repo root) ───────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "EDA" / "outputs_finals").exists():
    alt = PROJECT_ROOT.parent
    if (alt / "EDA" / "outputs_finals").exists():
        PROJECT_ROOT = alt
    else:
        raise FileNotFoundError(
            "Run this notebook from the project root (folder containing EDA/)."
        )

os.chdir(PROJECT_ROOT)
EDA_DIR = PROJECT_ROOT / "EDA"
FINALS_DIR = EDA_DIR / "outputs_finals"
FINDINGS_DIR = EDA_DIR / "aditya_findings"
CHARTS_DIR = FINDINGS_DIR / "outputs" / "charts"
REC_OUT = FINDINGS_DIR / "recommendation_systems" / "outputs"
REC_B_OUT = FINDINGS_DIR / "Recommendation_B" / "outputs"

sys.path.insert(0, str(FINDINGS_DIR))
from _shared import (  # noqa: E402
    CHART_STYLE,
    assign_decile,
    attach_true_profit,
    load_decile_pool,
    load_lines_with_margin,
    sku_label,
)

plt.rcParams.update(CHART_STYLE)
sns.set_theme(style="whitegrid")

print("Project root:", PROJECT_ROOT)
print("Finals dir:", FINALS_DIR)
print("Findings dir:", FINDINGS_DIR)


Project root: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505
Finals dir: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505\EDA\outputs_finals
Findings dir: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505\EDA\aditya_findings


## 1. Prerequisites & data validation

In [2]:
manifest_path = FINALS_DIR / "manifest.json"
assert manifest_path.exists(), f"Missing {manifest_path}"

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

required_parquets = ["customers.parquet", "orders.parquet", "lines.parquet"]
missing = [p for p in required_parquets if not (FINALS_DIR / p).exists()]
assert not missing, f"Missing parquet files: {missing}"

cogs_file = FINDINGS_DIR / "20260616-COGS_Data_Request_LushProtein (1).xlsx"
assert cogs_file.exists(), f"Missing COGS file: {cogs_file}"

decile_table = FINALS_DIR / "decile_customer_table.csv"
if not decile_table.exists():
    decile_table = EDA_DIR / "outputs" / "decile_customer_table.csv"
assert decile_table.exists(), (
    "Missing decile_customer_table.csv — run EDA/lushprotein_decile.ipynb first "
    "(Section that exports decile_customer_table.csv)."
)

expected = manifest["row_counts"]["primary_all_layers"]
print("Manifest row counts (primary_all_layers):")
for k, v in expected.items():
    print(f"  {k}: {v:,}")

customers = pd.read_parquet(FINALS_DIR / "customers.parquet")
orders = pd.read_parquet(FINALS_DIR / "orders.parquet")
lines = load_lines_with_margin()

assert len(customers) == expected["customers.parquet"], "Customer count mismatch vs manifest"
assert len(orders) == expected["orders.parquet"], "Order count mismatch vs manifest"
assert len(lines) == expected["lines.parquet"], "Line count mismatch vs manifest"
assert "true_gross_profit" in customers.columns, "Run enrich_finals_with_margin.py if true_gross_profit missing"

fc = customers[customers["finals_eligible"] == True].copy()
print(f"\nFinals-eligible customers loaded: {len(fc):,}")
print("Filters applied:", json.dumps(manifest["filters_applied"], indent=2)[:500], "...")


Manifest row counts (primary_all_layers):
  orders.parquet: 8,955
  lines.parquet: 14,448
  customers.parquet: 5,694
  lines_sku_analysis.parquet: 14,448


ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

## 2. Problem statement — retention & category ladder

These metrics drive Slide 1 of the Solution 2 presentation. Logic mirrors `EDA/aditya_findings/calc_slide_data.py`.


In [ ]:
one_and_done = (fc["finals_orders"] == 1).sum()
one_and_done_pct = one_and_done / len(fc) * 100
repeat_pct = (fc["finals_orders"] > 1).mean() * 100
single_cat_pct = (fc["n_categories_ever"] == 1).mean() * 100

print("=== PROBLEM STATEMENT ===")
print(f"Finals-eligible customers: {len(fc):,}")
print(f"One-and-done (1 order): {one_and_done:,} ({one_and_done_pct:.1f}%)")
print(f"Repeat customers (>1 order): {(fc['finals_orders'] > 1).sum():,} ({repeat_pct:.1f}%)")
print(f"Single-category buyers: {(fc['n_categories_ever'] == 1).sum():,} ({single_cat_pct:.1f}%)")

cat_ladder = (
    fc.groupby("n_categories_ever")
    .agg(
        n=("customer_id", "count"),
        avg_gp=("true_gross_profit", "mean"),
        repeat_rate=("finals_orders", lambda x: (x > 1).mean()),
        avg_orders=("finals_orders", "mean"),
    )
    .round(3)
)
display(cat_ladder)

sub = fc[fc["ever_subscribed"] == True]
nonsub = fc[fc["ever_subscribed"] == False]
sub_repeat = (sub["finals_orders"] > 1).mean() * 100
nonsub_repeat = (nonsub["finals_orders"] > 1).mean() * 100
print(f"\nSubscribers (n={len(sub):,}): repeat {sub_repeat:.1f}%")
print(f"Non-subscribers (n={len(nonsub):,}): repeat {nonsub_repeat:.1f}%")

# Presentation anchors (allow small drift if data refreshed)
assert 75 <= one_and_done_pct <= 80, f"Unexpected one-and-done rate: {one_and_done_pct:.1f}%"
assert 60 <= single_cat_pct <= 70, f"Unexpected single-category rate: {single_cat_pct:.1f}%"
print("\n✓ Key problem metrics within expected presentation ranges")


## 3. Layer 1 — Rule-based cold start (category co-purchase)

In [ ]:
CATEGORIES = [
    "Clear Protein", "Lean Protein", "Collagen Glow",
    "Accessories", "Soy Protein", "Other", "Unknown",
]

def build_co_purchase_matrix(line_df: pd.DataFrame) -> pd.DataFrame:
    cust_cats = line_df.groupby("customer_id")["product_category"].apply(set).reset_index()
    matrix = pd.DataFrame(index=CATEGORIES, columns=CATEGORIES, dtype=float)
    for row_cat in CATEGORIES:
        buyers = cust_cats[cust_cats["product_category"].apply(lambda s: row_cat in s)]
        n = len(buyers)
        if n == 0:
            continue
        for col_cat in CATEGORIES:
            if row_cat == col_cat:
                matrix.loc[row_cat, col_cat] = 100.0
            else:
                also = buyers["product_category"].apply(lambda s: col_cat in s).sum()
                matrix.loc[row_cat, col_cat] = round(also / n * 100, 1)
    return matrix

co_all = build_co_purchase_matrix(lines)
pool = attach_true_profit(load_decile_pool(), lines)
pool["profit_decile"] = assign_decile(pool["true_gross_profit"])
d1_ids = set(pool.loc[pool["profit_decile"] == "D1", "customer_id"])
co_d1 = build_co_purchase_matrix(lines[lines["customer_id"].isin(d1_ids)])

print("All-pool co-purchase (% of row-category buyers who also bought column category):")
display(co_all.loc[["Clear Protein", "Lean Protein", "Collagen Glow"], ["Clear Protein", "Lean Protein", "Collagen Glow", "Accessories"]])

print("\nHero cross-sells (all pool):")
print(f"  Clear → Lean: {co_all.loc['Clear Protein', 'Lean Protein']:.1f}%")
print(f"  Lean → Clear: {co_all.loc['Lean Protein', 'Clear Protein']:.1f}%")
print(f"  Accessories → Lean: {co_all.loc['Accessories', 'Lean Protein']:.1f}%")

# Layer 1 rules exported by build_recommenders.py
RULES = [
    {"if_bought": "Clear Protein", "recommend": "Lean Protein"},
    {"if_bought": "Lean Protein", "recommend": "Clear Protein"},
    {"if_bought": "Lean Protein", "recommend": "Accessories"},
    {"if_bought": "Clear Protein", "recommend": "Accessories"},
    {"if_bought": "Collagen Glow", "recommend": "Clear Protein"},
    {"if_bought": "Accessories", "recommend": "Lean Protein"},
    {"if_bought": "Accessories", "recommend": "Clear Protein"},
]
pd.DataFrame(RULES)


## 4. Layer 2 — Market basket analysis (same-order SKU pairs)

In [ ]:
from itertools import combinations

# Inline MBA (same logic as recommendation_systems/sku_market_basket.py)
lines_mba = lines.copy()
lines_mba["sku_display"] = lines_mba.apply(sku_label, axis=1)

baskets = (
    lines_mba.groupby("order_id")["sku_display"]
    .apply(lambda s: sorted(set(s)))
    .reset_index()
)
n_orders = len(baskets)
item_counts = lines_mba.groupby("sku_display")["order_id"].nunique()
pair_counts = {}

for items in baskets["sku_display"]:
    if len(items) < 2:
        continue
    for a, b in combinations(items, 2):
        key = (a, b) if a < b else (b, a)
        pair_counts[key] = pair_counts.get(key, 0) + 1

MIN_SUPPORT, MIN_CONFIDENCE = 0.005, 0.05
rules = []
for (a, b), cnt in pair_counts.items():
    support = cnt / n_orders
    if support < MIN_SUPPORT:
        continue
    conf_ab = cnt / item_counts.get(a, 1)
    conf_ba = cnt / item_counts.get(b, 1)
    if conf_ab >= MIN_CONFIDENCE:
        rules.append({
            "antecedent": a, "consequent": b,
            "support": round(support, 4), "confidence": round(conf_ab, 4),
            "lift": round(conf_ab / (item_counts.get(b, 1) / n_orders), 2) if item_counts.get(b, 0) > 0 else np.nan,
        })
    if conf_ba >= MIN_CONFIDENCE:
        rules.append({
            "antecedent": b, "consequent": a,
            "support": round(support, 4), "confidence": round(conf_ba, 4),
            "lift": round(conf_ba / (item_counts.get(a, 1) / n_orders), 2) if item_counts.get(a, 0) > 0 else np.nan,
        })

rules_df = (
    pd.DataFrame(rules)
    .sort_values(["confidence", "support"], ascending=False)
    .drop_duplicates(subset=["antecedent", "consequent"])
)
top_rules = rules_df.head(15)
print(f"Orders analysed: {n_orders:,} | Rules passing threshold: {len(rules_df):,}")
display(top_rules)

if len(top_rules):
    r0 = top_rules.iloc[0]
    print(f"\nTop rule: {r0['antecedent'][:50]} → {r0['consequent'][:50]} (conf={r0['confidence']:.2f})")


## 5. Layer 3 — Timed post-purchase cross-sell & samples

In [ ]:
REORDER_MEDIAN_DAYS = {
    "Clear Protein": 54,
    "Lean Protein": 35,
    "Collagen Glow": 42,
    "Soy Protein": 84,
    "Accessories": 35,
    "Other": 50,
    "Unknown": 50,
}

CROSS_CATEGORY_MAP = {
    "Clear Protein": {"recommend_category": "Lean Protein", "sample_category": "Collagen Glow", "email_day": 14, "evidence_pct": 25.0},
    "Lean Protein": {"recommend_category": "Clear Protein", "sample_category": "Collagen Glow", "email_day": 14, "evidence_pct": 28.9},
    "Collagen Glow": {"recommend_category": "Clear Protein", "sample_category": "Lean Protein", "email_day": 21, "evidence_pct": 23.8},
    "Accessories": {"recommend_category": "Lean Protein", "sample_category": "Clear Protein", "email_day": 7, "evidence_pct": 41.2},
    "Soy Protein": {"recommend_category": "Clear Protein", "sample_category": "Lean Protein", "email_day": 14, "evidence_pct": 10.0},
    "Other": {"recommend_category": "Clear Protein", "sample_category": "Lean Protein", "email_day": 14, "evidence_pct": 15.5},
    "Unknown": {"recommend_category": "Clear Protein", "sample_category": "Lean Protein", "email_day": 14, "evidence_pct": 9.1},
}

timing_rows = []
for cat, spec in CROSS_CATEGORY_MAP.items():
    reorder = REORDER_MEDIAN_DAYS.get(cat, 50)
    email_day = spec["email_day"]
    sample_ship_day = max(email_day + 7, reorder - 10)
    timing_rows.append({
        "first_product_category": cat,
        "cross_sell_category": spec["recommend_category"],
        "sample_category": spec["sample_category"],
        "email_cross_sell_day_after_delivery": email_day,
        "physical_sample_day_after_delivery": sample_ship_day,
        "median_reorder_days": reorder,
        "co_purchase_evidence_pct": spec["evidence_pct"],
    })

timing_df = pd.DataFrame(timing_rows)
display(timing_df)

# Compare to saved output if present
saved_timing = FINDINGS_DIR / "outputs" / "cross_sell_timing_and_samples.csv"
if saved_timing.exists():
    saved = pd.read_csv(saved_timing)
    key_cols = [
        "first_product_category", "physical_sample_day_after_delivery", "median_reorder_days"
    ]
    merged = timing_df[key_cols].merge(saved[key_cols], on="first_product_category", suffixes=("_nb", "_saved"))
    assert (merged["physical_sample_day_after_delivery_nb"] == merged["physical_sample_day_after_delivery_saved"]).all()
    print("\n✓ Layer 3 timing table matches saved CSV")
else:
    print("\n(saved CSV not found yet — run Section 8 to generate)")


## 6. Layer 4 — Item-item collaborative filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

lines_cf = lines.copy()
lines_cf["sku_display"] = lines_cf.apply(sku_label, axis=1)
cust_sku = lines_cf.groupby(["customer_id", "sku_display"]).size().unstack(fill_value=0)
cust_sku = (cust_sku > 0).astype(int)

min_customers = 10
sku_counts = cust_sku.sum(axis=0)
keep = sku_counts[sku_counts >= min_customers].index.tolist()
mat = cust_sku[keep]

item_sim = pd.DataFrame(cosine_similarity(mat.T), index=keep, columns=keep)

def cf_recommend(bought_skus, top_n=5):
    scores = {}
    for sku in bought_skus:
        if sku not in item_sim.index:
            continue
        sims = item_sim.loc[sku].drop(bought_skus, errors="ignore")
        for other, score in sims.items():
            scores[other] = scores.get(other, 0) + score
    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]

hero = "clear-protein|Peach"
if hero in item_sim.index:
    recs = cf_recommend([hero])
    print(f"Item-CF recommendations after buying {hero}:")
    for sku, score in recs:
        print(f"  {sku}: similarity score {score:.3f}")
else:
    print(f"Hero SKU '{hero}' not in CF matrix; showing top SKUs by buyer count:")
    print(sku_counts.sort_values(ascending=False).head(8))

print(f"\nCF matrix: {item_sim.shape[0]} SKUs × {item_sim.shape[1]} SKUs (min {min_customers} buyers)")


## 7. CRM tiers & business value (Layer 3 targeting)

In [ ]:
tier_grp = (
    fc.groupby("crm_tier")
    .agg(
        n=("customer_id", "count"),
        avg_gp=("true_gross_profit", "mean"),
        avg_orders=("finals_orders", "mean"),
        repeat_rate=("finals_orders", lambda x: (x > 1).mean()),
    )
    .round(2)
)
print("CRM tier breakdown (from enriched finals customers):")
display(tier_grp)

# Tier progression value (presentation scenario)
fc_sorted = fc.sort_values("true_gross_profit", ascending=False)
next858 = fc_sorted.iloc[632:1490]  # Silver band
next264 = fc_sorted.iloc[368:632]   # Gold-B band
lift = next264["true_gross_profit"].mean() - next858["true_gross_profit"].mean()
conv_5 = int(len(next858) * 0.05)
print(f"\nSilver → Gold-B GP lift: +S${lift:.0f}/customer")
print(f"5% Silver conversion scenario: {conv_5} customers × S${lift:.0f} = S${conv_5 * lift:,.0f} incremental GP")

crm_path = FINDINGS_DIR / "outputs" / "crm_treatment_tiers.csv"
if crm_path.exists():
    crm = pd.read_csv(crm_path)
    print(f"\nSaved CRM tiers: {len(crm):,} customers | tiers: {crm['crm_treatment_tier'].value_counts().to_dict()}")


## 8. Regenerate Solution 2 outputs (optional batch run)

Runs the same Python scripts used during analysis. Skip this cell if you only need to verify numbers inline above.

**Order:** Recommendation B → MBA → 4-layer export → CRM/timing → rec-sys charts.


In [ ]:
RUN_REGENERATE = False  # set True to rerun all Solution 2 .py scripts and refresh CSVs/charts

SOLUTION2_SCRIPTS = [
    FINDINGS_DIR / "Recommendation_B" / "run_recommendation_b.py",
    FINDINGS_DIR / "recommendation_systems" / "sku_market_basket.py",
    FINDINGS_DIR / "recommendation_systems" / "build_recommenders.py",
    FINDINGS_DIR / "build_crm_tiers_and_timing.py",
    FINDINGS_DIR / "build_rec_sys_charts.py",
]

if RUN_REGENERATE:
    for script in SOLUTION2_SCRIPTS:
        if not script.exists():
            raise FileNotFoundError(script)
        print("=" * 60)
        print("Running", script.relative_to(PROJECT_ROOT))
        rc = subprocess.call([sys.executable, str(script)], cwd=str(PROJECT_ROOT))
        if rc != 0:
            raise RuntimeError(f"Script failed ({rc}): {script.name}")
    print("\n✓ All Solution 2 scripts completed")
else:
    print("Skipped regeneration (RUN_REGENERATE=False)")


## 9. Output manifest

In [ ]:
output_files = {
    "Layer 1 co-purchase (all)": REC_B_OUT / "co_purchase_matrix_all.csv",
    "Layer 1 co-purchase (D1)": REC_B_OUT / "co_purchase_matrix_d1.csv",
    "Layer 1 rules": REC_OUT / "recommender_01_rule_based.csv",
    "Layer 2 MBA top rules": REC_OUT / "sku_association_rules.csv",
    "Layer 2 sequential 1→2": REC_OUT / "first_to_second_sku_matrix.csv",
    "Layer 3 timing matrix": FINDINGS_DIR / "outputs" / "cross_sell_timing_and_samples.csv",
    "Layer 3 sample strategy": FINDINGS_DIR / "outputs" / "sample_strategy_by_purchase_stage.csv",
    "Layer 4 item similarity": REC_OUT / "recommender_04_item_similarity_matrix.csv",
    "CRM treatment tiers": FINDINGS_DIR / "outputs" / "crm_treatment_tiers.csv",
}

rows = []
for label, path in output_files.items():
    rows.append({
        "artifact": label,
        "path": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
    })

manifest_df = pd.DataFrame(rows)
display(manifest_df)

missing_out = manifest_df[~manifest_df["exists"]]
if len(missing_out):
    print("\nMissing outputs — run Section 8 with RUN_REGENERATE=True")
else:
    print("\n✓ All expected CSV artifacts present")


## 10. Presentation figures

In [ ]:
KEY_CHARTS = [
    ("Problem statement", CHARTS_DIR / "slide1_problem_statement.png"),
    ("Four-layer architecture", CHARTS_DIR / "rec_sys_why_4_layers.png"),
    ("Layer 1 cold start", CHARTS_DIR / "rec_sys_l1_cold_start.png"),
    ("Layer 2 MBA", CHARTS_DIR / "rec_sys_l2_mba_rules.png"),
    ("Layer 3 timing", CHARTS_DIR / "rec_sys_l3_timing_detail.png"),
    ("Layer 4 item-CF", CHARTS_DIR / "rec_sys_l4_item_cf_heatmap.png"),
    ("Category ladder", CHARTS_DIR / "r2_category_ladder.png"),
    ("Cross-sell timeline (Clear)", CHARTS_DIR / "r2_cross_sell_timeline_clear.png"),
]

for title, path in KEY_CHARTS:
    print(title)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"  (missing — run Section 8: {path.name})")
    print()


---

**Reproducibility checklist**
- [ ] Notebook run from project root without errors/ path edits
- [ ] `manifest.json` row counts match loaded parquets
- [ ] Problem metrics (~77% one-and-done, ~65% single-category) reproduced in Section 2
- [ ] Layer 3 timing matches `cross_sell_timing_and_samples.csv` (7 rows)
- [ ] Section 8 regenerates CSVs + charts used in presentation
- [ ] Section 10 displays all key PNGs

**Source scripts:** `EDA/aditya_findings/run_all.py` runs the full pipeline including margin analysis and pitch charts.
